# Order Finding

## Given $a$ coprime to $N$, the **order** of $a$ modulo $N$ is the smallest positive integer $r$ with $a^r \equiv 1 \pmod N$. Order finding is the quantum subroutine that powers Shor's factoring algorithm. Reducing factoring to order finding is where Shor's genius lies; the *quantum* part is then just QPE applied to the modular multiplier.

# 1. From order finding to phase estimation

## The unitary $M_a : |y\rangle \mapsto |a y \bmod N\rangle$ has eigenvectors

$$ \Large  |u_s\rangle = \frac{1}{\sqrt r} \sum_{k=0}^{r-1} e^{-2\pi i\, s k / r}\, |a^k \bmod N\rangle, \qquad s = 0, 1, \dots, r-1, $$

## with eigenvalues $e^{2\pi i\, s / r}$. Phase estimation on $M_a$ with input $|u_s\rangle$ would return $s/r$. We don't know any $|u_s\rangle$ explicitly, but the equal superposition over $s$ is just $|1\rangle$:

$$ \Large  \frac{1}{\sqrt r} \sum_{s=0}^{r-1} |u_s\rangle = |1\rangle. $$

## So feeding QPE the easy-to-prepare state $|1\rangle$ gives a random $s$ with probability $1/r$ each, and the bits read out are a noisy approximation to $s/r$.

# 2. Extracting $r$ from $s/r$

## The measured bitstring gives an integer $k$ close to $s \cdot 2^n / r$. Run the **continued-fractions** algorithm on $k / 2^n$ to find a fraction $s/r$ with $r < N$. If the resulting $r$ satisfies $a^r \equiv 1 \pmod N$, we are done. Otherwise rerun and combine the two outputs.

# 3. Putting the pieces together

## Algorithm (probabilistic):

## 1. Pick $a$ coprime to $N$.
## 2. Run phase estimation on $M_a$ with input register in $|1\rangle$ to obtain bitstring $k$.
## 3. Use continued fractions on $k / 2^n$ to obtain a candidate $r$.
## 4. Check that $a^r \equiv 1 \pmod N$. If yes, output $r$; otherwise repeat.

## The success probability per run is constant (after polynomially many tries), so the expected number of rounds is $O(1)$ with high probability.

In [ ]:
# A pure-classical simulation of order finding (no quantum here), useful to see the structure.
from fractions import Fraction
import math

def order_classical(a, N):
    r = 1
    x = a % N
    while x != 1:
        x = (x * a) % N
        r += 1
    return r

for a, N in [(7, 15), (11, 21), (2, 35)]:
    r = order_classical(a, N)
    print(f'order of {a} mod {N} = {r}  (check: {a}^{r} mod {N} = {pow(a, r, N)})')

In [ ]:
# Demonstrate the continued-fraction post-processing on an idealised QPE outcome.
# Suppose QPE returned k = 192, with n = 8 counting qubits and N = 15.
n = 8
k = 192
approx = Fraction(k, 2**n)
print(f'k / 2^n = {approx}, decimal = {float(approx):.6f}')
# Continued-fraction expansion truncated to denominators < N = 15
best = None
for cf in [approx.limit_denominator(d) for d in range(2, 15)]:
    print(f'  best fraction with denom <= {cf.denominator}: {cf}')

# 4. Why this is the heart of Shor

## Factoring $N$ reduces (classically, with high probability) to:

## - Pick random $a$ with $\gcd(a, N) = 1$.
## - Find the order $r$ of $a$ mod $N$.
## - If $r$ is even and $a^{r/2} \not\equiv -1 \pmod N$, then $\gcd(a^{r/2} \pm 1, N)$ are nontrivial factors of $N$.

## All of this except *find the order* is fast classically. The order-finding step is where quantum buys the exponential speedup.

# Recap

## - Order finding = phase estimation on the modular multiplier $M_a$.
## - Output: an approximation to $s/r$; recover $r$ by continued fractions.
## - Reduces factoring to a polynomially many quantum runs.

## Next: the full Shor algorithm assembled.